# Part III-c registered cost tables: Llama-3.1-8B (Colab A100 80 GB)

The registered prereg's amendment (`a1f1301`, 2026-09-30) runs the 8B on a Colab A100 80 GB. This notebook runs
the pinned harness's cost-table phase, exactly as an NRP job would:

- harness commit `a1f13010bf1d`, torch 2.8.0 + torchvision 0.23.0, transformers 4.56.1
- model `unsloth/Meta-Llama-3.1-8B` at `e9a141a2091e` (checked)
- the text and the 128 calibration / 48 scored windows checked against the prereg's recorded sha256
- gate G0 on this GPU, then `codec_run tables`

**Results go to Google Drive**, `MyDrive/tqp-weights/codec/llama3.1-8b/a1f13010bf1d`, **one matrix at a time**. If Colab
disconnects, reconnect to an A100 80 GB and Run all again: finished matrices are skipped (224 in all).

**Runtime: A100 80 GB.** Expected: ~10 min of setup, then roughly 2-3 hours of GPU work.

In [ ]:
# 1. GPU: the A100 80 GB (the cost tables peak at ~33 GiB, but the prereg names this product).
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU (A100)"
gib = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"{gib:.1f} GiB"); assert gib > 70, "not the 80 GB A100: reconnect until the first cell shows ~79 GiB"

In [ ]:
# 1b. Google Drive: results in MyDrive/tqp-weights/codec/llama3.1-8b/a1f13010bf1d (G: on the laptop).
from google.colab import drive
drive.mount("/content/drive")
import os
RESULTS = "/content/drive/MyDrive/tqp-weights/codec/llama3.1-8b/a1f13010bf1d"
os.makedirs(RESULTS, exist_ok=True); print("results ->", RESULTS)

In [ ]:
# 2. The pinned environment (torch and torchvision must match).
!pip -q install "torch==2.8.0" "torchvision==0.23.0" "transformers==4.56.1" "accelerate==1.14.0" safetensors numpy huggingface_hub
!python -c "import torch, transformers; print('pinned:', torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))"

In [ ]:
# 3. The harness at its pinned commit.
import os, subprocess
C = "a1f13010bf1dfed3251b73be1e945079c8d8af42"
if not os.path.exists("/content/tqp"):
    subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--no-checkout",
                    "https://github.com/ahb-sjsu/turboquant-pro", "/content/tqp"], check=True)
    subprocess.run(["git", "-C", "/content/tqp", "sparse-checkout", "set", "--no-cone", "benchmarks/weight_observer"], check=True)
    subprocess.run(["git", "-C", "/content/tqp", "checkout", "-q", C], check=True)
head = subprocess.run(["git", "-C", "/content/tqp", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == C, head
os.environ.update(PYTHONPATH="/content/tqp/benchmarks", PYTHONUNBUFFERED="1", HF_HUB_DISABLE_XET="1",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", TOKENIZERS_PARALLELISM="false")
print("harness", head)
def step(args, label):
    """Run one pinned command, streaming its output here (Colab hides a child's own output)."""
    print(">>", label, flush=True)
    p = subprocess.Popen(["python", "-m", *args], cwd="/content/tqp/benchmarks",
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        lines.append(line)
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"step failed ({p.returncode}): {label}" + chr(10) + "".join(lines[-30:]))


In [ ]:
# 4. The text, restaged and checked against the prereg.
import hashlib
step(["weight_observer.stage_text", "--dest", "/content/wo/text"], "stage_text")
PIN = {"train.txt": "aee724fa58bfbdeb3fc6803297fb6bab27b203d7c40b39ddef9b9770e5d52fe5",
       "test.txt": "696cca6b65a171b0a358a4be6732cdfdf2dd6164a32e20fd70e3c13fc4dfae83"}
for n, h in PIN.items():
    got = hashlib.sha256(open(f"/content/wo/text/{n}", "rb").read()).hexdigest()
    assert got == h, (n, got); print(n, "matches the prereg")

In [ ]:
# 5. The model at the prereg's mirror commit (16.1 GB), manifest checked.
import json
M = "/content/wo/models/llama3.1-8b"
step(["weight_observer.stage_models", "--model-id", "unsloth/Meta-Llama-3.1-8B", "--revision", "e9a141a2091ea561b96483212645a2a05e6f99fc", "--dest", M], "stage_models")
assert json.load(open(f"{M}/STAGED.json"))["revision"] == "e9a141a2091ea561b96483212645a2a05e6f99fc"; print("staged at the prereg's commit")

In [ ]:
# 6. The windows, tokenized once and checked against the registered hashes.
W = "/content/wo/windows/llama3.1-8b"
if not os.path.exists(f"{W}/windows.pt"):
    step(["weight_observer.codec_run", "windows", "--model-path", M, "--text", "/content/wo/text", "--out", W], "windows")
h = json.load(open(f"{W}/hashes.json"))
assert h["calibration_windows"]["sha256"] == "7dcba79b76b0e185ab48d7e27697db01da6345122b6fd0f000a3d64ae9229db3", "calibration windows differ from the prereg"
assert h["evaluation_windows"]["sha256"] == "2eda697bba5e21a5f25b95c50528b3325025ffafc2c7ef020a1e2211af163223", "scored windows differ from the prereg"
print("windows match the registered hashes")

In [ ]:
# 7. Gate G0 on this GPU, then the cost tables (resumable: finished matrices are skipped).
step(["weight_observer.g0_device", "--model-path", M, "--out", RESULTS], "G0")
step(["weight_observer.codec_run", "tables", "--model-key", "llama3.1-8b", "--model-path", M, "--text", "/content/wo/text",
      "--windows", f"{W}/windows.pt", "--out", RESULTS], "cost tables")

In [ ]:
# 8. Check and hand-back.
rows = open(f"{RESULTS}/codec_costs.jsonl").read().splitlines()
g0 = json.load(open(f"{RESULTS}/g0_device.json"))
print("matrices:", len(rows), "of 224 | G0 passed:", g0["passed"], "on", g0["device"])
assert len(rows) == 224 and g0["passed"]
drive.flush_and_unmount()
print("DONE: MyDrive/tqp-weights/codec/llama3.1-8b (G: on the laptop)")